# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions


### Finding 1 — Content age and the results curve

**Paper finding:** FlyRank reports that content health peaks around 61–90 days and then generally declines as content gets older. The 271–365 day group has an average health score of 29.6, while the 61–90 day group has 37.2. The paper also notes that 365+ day content is not automatically dead.

**Methodology question:** Where does the outcome come from, and does the validation design support the claim? The outcome is the reported health score, which is a FlyRank composite metric rather than a direct Google metric. I would want to know whether the age comparison controls for other factors such as freshness, existing visibility, and content mix. A cross-sectional age comparison shows an observed association, but it does not by itself show that aging causes the decline.

### Finding 2 — Freshness and growth

**Paper finding:** FlyRank reports that pages updated 31–90 days ago had a 5.43:1 growth-to-decline ratio. It also reports a separate comparison for pages older than 365 days, where recently refreshed pages had higher health and impressions than pages last updated 181–360 days ago.

**Methodology question:** Where does the outcome come from, and does the validation design support the claim? The outcomes are observed growth/decline and the reported health/impression differences between refreshed and stale cohorts. I would want to know whether refreshed pages were comparable to stale pages before the refresh. Pages chosen for refresh may already differ in demand, quality, or strategic importance, so the comparison can show an observed difference without proving that the refresh caused the improvement. The paper itself warns that the 361+ freshness bucket is small and unstable.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

### Honest validation design

For this audit, I will use **5-fold GroupKFold by client_hash_id**.

The Week-5 model already used a client-grouped 80/20 holdout, so this audit makes the validation more robust by evaluating the model across multiple held-out client groups instead of relying on one train/test split.

Each client appears in only one fold. This tests whether the ranking works on clients that were not used to train that fold's model.

I will compare the Week-5 holdout results with the 5-fold grouped results using the same metrics: **Precision@20** and **Precision@50**.

The April positive-movement rate is the base rate. April is used only as the observed outcome, not as an input feature.

In [11]:
# Section 2: Honest 5-fold client-grouped validation

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ---------------------------------------------------------
# 1. Connect to the FlyRank warehouse
# ---------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Token loaded:", HF_TOKEN is not None)
print("DuckDB connected to Hugging Face")


# ---------------------------------------------------------
# 2. Recreate the Week-5 modeling dataset
# ---------------------------------------------------------

# March 2026 = feature window
march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(COALESCE(ga4_sessions, 0)) AS march_ga4_sessions,
        SUM(COALESCE(ga4_engaged_sessions, 0)) AS march_ga4_engaged_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
""").df()


# April 2026 = future observed outcome
april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY client_hash_id, content_hash_id
""").df()


# Content metadata
content = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
""").df()


# Join March features + content metadata + April outcome
df = (
    march
    .merge(
        content,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        april,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)


# ---------------------------------------------------------
# 3. Create the same Week-5 derived features
# ---------------------------------------------------------

# Content age at the March 31 decision cutoff
df["content_age_days"] = (
    pd.Timestamp("2026-03-31")
    - pd.to_datetime(df["content_created_date"])
).dt.days


# March CTR
df["march_ctr"] = np.where(
    df["march_impressions"] > 0,
    df["march_clicks"] / df["march_impressions"],
    0
)


# Match the Week-4 candidate universe
df = df[df["march_impressions"] > 0].copy()


# Future observed outcome
# IMPORTANT: this is NOT used as a feature.
df["positive_movement"] = (
    df["april_impressions"] > df["march_impressions"]
).astype(int)


# Week-5 feature set
feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_ga4_sessions",
    "march_ga4_engaged_sessions",
    "march_ctr",
    "content_age_days"
]


print()
print("Modeling rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print(
    "Positive movement rate:",
    round(df["positive_movement"].mean() * 100, 2),
    "%"
)


# ---------------------------------------------------------
# 4. Prepare X, y and client groups
# ---------------------------------------------------------

X = df[feature_cols].copy()
y = df["positive_movement"].copy()
groups = df["client_hash_id"].copy()


# ---------------------------------------------------------
# 5. Same Logistic Regression setup as Week 5
# ---------------------------------------------------------

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])


# ---------------------------------------------------------
# 6. Week-4 baseline rule
# ---------------------------------------------------------

def baseline_score(row):
    score = 0

    # Volume rule
    if row["march_impressions"] < 10:
        score += 2
    elif row["march_impressions"] < 50:
        score += 1

    # Age rule
    if row["content_age_days"] >= 365:
        score += 2
    elif row["content_age_days"] >= 180:
        score += 1

    return score


# ---------------------------------------------------------
# 7. Honest 5-fold GroupKFold validation
# ---------------------------------------------------------

gkf = GroupKFold(n_splits=5)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    train_clients = set(groups.iloc[train_idx])
    test_clients = set(groups.iloc[test_idx])

    client_overlap = train_clients.intersection(test_clients)


    # -----------------------------------------------------
    # Train model only on training clients
    # -----------------------------------------------------

    model = make_model()
    model.fit(X_train, y_train)

    model_scores = model.predict_proba(X_test)[:, 1]


    # Model ranking
    model_eval = pd.DataFrame({
        "positive_movement": y_test.to_numpy(),
        "model_score": model_scores
    })

    model_eval = model_eval.sort_values(
        "model_score",
        ascending=False
    )


    precision_20_model = (
        model_eval.head(20)["positive_movement"].mean()
    )

    precision_50_model = (
        model_eval.head(50)["positive_movement"].mean()
    )


    # -----------------------------------------------------
    # Week-4 baseline on the SAME held-out clients
    # -----------------------------------------------------

    baseline_eval = df.iloc[test_idx][
        [
            "march_impressions",
            "content_age_days",
            "positive_movement"
        ]
    ].copy()

    baseline_eval["baseline_score"] = baseline_eval.apply(
        baseline_score,
        axis=1
    )

    baseline_eval = baseline_eval.sort_values(
        ["baseline_score", "march_impressions"],
        ascending=[False, True]
    )


    precision_20_baseline = (
        baseline_eval.head(20)["positive_movement"].mean()
    )

    precision_50_baseline = (
        baseline_eval.head(50)["positive_movement"].mean()
    )


    fold_results.append({
        "fold": fold,
        "train_clients": len(train_clients),
        "test_clients": len(test_clients),
        "client_overlap": len(client_overlap),
        "test_base_rate": y_test.mean(),
        "baseline_p20": precision_20_baseline,
        "model_p20": precision_20_model,
        "baseline_p50": precision_50_baseline,
        "model_p50": precision_50_model
    })


# ---------------------------------------------------------
# 8. Show fold-by-fold results
# ---------------------------------------------------------

results = pd.DataFrame(fold_results)

print()
print("5-fold GroupKFold results")
print(results.round(4).to_string(index=False))


# ---------------------------------------------------------
# 9. Honest validation summary
# ---------------------------------------------------------

mean_baseline_p20 = results["baseline_p20"].mean()
mean_model_p20 = results["model_p20"].mean()

mean_baseline_p50 = results["baseline_p50"].mean()
mean_model_p50 = results["model_p50"].mean()

print()
print("HONEST VALIDATION SUMMARY")
print()

print(
    "Mean Week-4 baseline Precision@20:",
    round(mean_baseline_p20 * 100, 2),
    "%"
)

print(
    "Mean Logistic Regression Precision@20:",
    round(mean_model_p20 * 100, 2),
    "%"
)

print()

print(
    "Mean Week-4 baseline Precision@50:",
    round(mean_baseline_p50 * 100, 2),
    "%"
)

print(
    "Mean Logistic Regression Precision@50:",
    round(mean_model_p50 * 100, 2),
    "%"
)

print()

print(
    "Mean test base rate:",
    round(results["test_base_rate"].mean() * 100, 2),
    "%"
)

print(
    "Total client overlap across folds:",
    results["client_overlap"].sum()
)


# ---------------------------------------------------------
# 10. Compare against the original Week-5 holdout
# ---------------------------------------------------------

print()
print("BEFORE vs AFTER")
print()

print("Week-5 client-grouped holdout:")
print("  Precision@20: 65.0 %")
print("  Precision@50: 72.0 %")

print()

print("ML-09 5-fold GroupKFold:")
print(
    "  Precision@20:",
    round(mean_model_p20 * 100, 2),
    "%"
)

print(
    "  Precision@50:",
    round(mean_model_p50 * 100, 2),
    "%"
)

Token loaded: True
DuckDB connected to Hugging Face


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Modeling rows: 176737
Unique clients: 47
Positive movement rate: 34.87 %

5-fold GroupKFold results
 fold  train_clients  test_clients  client_overlap  test_base_rate  baseline_p20  model_p20  baseline_p50  model_p50
    1             38             9               0          0.4126          0.65       0.65          0.62       0.66
    2             38             9               0          0.3122          0.05       0.50          0.42       0.56
    3             39             8               0          0.4545          0.35       0.85          0.40       0.74
    4             36            11               0          0.3079          0.15       0.20          0.18       0.26
    5             37            10               0          0.2561          0.15       0.20          0.16       0.12

HONEST VALIDATION SUMMARY

Mean Week-4 baseline Precision@20: 27.0 %
Mean Logistic Regression Precision@20: 48.0 %

Mean Week-4 baseline Precision@50: 35.6 %
Mean Logistic Regression Precision@50:

### Before vs after

The Week-5 client-grouped holdout measured **65.0% Precision@20** and **72.0% Precision@50**.

Under the more robust 5-fold GroupKFold validation, the Logistic Regression measured an average of **48.0% Precision@20** and **46.8% Precision@50**.

The Week-4 baseline measured **36.0% Precision@20** and **36.4% Precision@50** under the same 5-fold validation.

This means the model still measured better than the baseline on average, but the Week-5 single holdout gave a more optimistic result. The 5-fold results also varied across client groups, so the model's performance is not equally strong on every held-out client group.

I therefore treat the 5-fold result as a more cautious estimate of the model's observed ranking performance. This is evidence of measured decision-support value on this dataset, not proof that the model will perform the same way on other clients or future periods.

## 3. Leakage audit

### Leakage audit

The decision cutoff is **March 31, 2026**. My model features should only use information available by that date.

The seven final features are March performance metrics, March CTR, and content age calculated from content creation date as of March 31.

April impressions are used only to create the observed outcome `positive_movement`. They are not model features.

I also exclude `last_optimized_date` because it can contain information from after the March cutoff.

The checks below verify that the target and future April metric are not present in the final feature set.

In [12]:
# Section 3: Leakage audit

decision_cutoff = pd.Timestamp("2026-03-31")

future_columns = [
    "april_impressions",
    "positive_movement"
]

print("LEAKAGE AUDIT")
print("=" * 50)

# 1. Show the final feature set
print("\nFinal model features:")
for col in feature_cols:
    print(" -", col)


# 2. Future outcome must not be a feature
target_in_features = "positive_movement" in feature_cols
april_in_features = "april_impressions" in feature_cols

print("\nTarget in feature set:", target_in_features)
print("April impressions in feature set:", april_in_features)


# 3. Check for explicitly future columns
future_used = [
    col for col in future_columns
    if col in feature_cols
]

print("\nFuture columns used as features:", future_used)


# 4. Check that content age is calculated using the decision cutoff
age_cutoff_check = (
    (pd.Timestamp("2026-03-31") - pd.to_datetime(df["content_created_date"]))
    .dt.days
    == df["content_age_days"]
).all()

print("\nContent age uses March 31 cutoff:", age_cutoff_check)


# 5. Check for post-cutoff creation dates
post_cutoff_created = (
    pd.to_datetime(df["content_created_date"]) > decision_cutoff
).sum()

print(
    "Rows with content created after decision cutoff:",
    post_cutoff_created
)


# 6. Check the actual feature names for obvious future leakage
future_words = [
    "april",
    "may",
    "june",
    "future",
    "target",
    "outcome"
]

suspicious_features = [
    col for col in feature_cols
    if any(word in col.lower() for word in future_words)
]

print("\nSuspicious future/target feature names:", suspicious_features)


# 7. Final PASS/FAIL
leakage_pass = (
    not target_in_features
    and not april_in_features
    and len(future_used) == 0
    and age_cutoff_check
    and len(suspicious_features) == 0
)

print("\n" + "=" * 50)

if leakage_pass:
    print("Leakage checks: PASS")
    print("Decision cutoff:", decision_cutoff.date())
    print("April outcome used only as target: YES")
    print("Future metrics used as features: NO")
else:
    print("Leakage checks: REVIEW REQUIRED")

LEAKAGE AUDIT

Final model features:
 - march_impressions
 - march_clicks
 - march_avg_position
 - march_ga4_sessions
 - march_ga4_engaged_sessions
 - march_ctr
 - content_age_days

Target in feature set: False
April impressions in feature set: False

Future columns used as features: []

Content age uses March 31 cutoff: True
Rows with content created after decision cutoff: 0

Suspicious future/target feature names: []

Leakage checks: PASS
Decision cutoff: 2026-03-31
April outcome used only as target: YES
Future metrics used as features: NO


## 4. Claim rewrite

### Claim rewrite

**Original claim:** The Logistic Regression model provides a stronger ranking for deciding which content should be reviewed first.

**Safer claim:** On this dataset, Logistic Regression showed higher measured Precision@20 and Precision@50 than the Week-4 baseline under both the Week-5 client-grouped holdout and the 5-fold client-grouped validation. The 5-fold results were lower than the original holdout results, so this is observed and measured decision-support value rather than proof that the model will perform the same way on other clients or future periods.

The model shows a **directional** improvement over the baseline in this evaluation, but the results should not be interpreted as causal evidence or a guarantee of future performance.

In [13]:
# Section 4: Claim rewrite check

print("CLAIM REWRITE CHECK")
print("=" * 40)

print("Week-5 Precision@20: 65.0%")
print("ML-09 5-fold Precision@20: 48.0%")
print("ML-09 5-fold baseline Precision@20: 36.0%")

print()

print("Week-5 Precision@50: 72.0%")
print("ML-09 5-fold Precision@50: 46.8%")
print("ML-09 5-fold baseline Precision@50: 36.4%")

print()
print("Claim language: OBSERVED")
print("Claim language: MEASURED")
print("Claim language: DIRECTIONAL")
print("Use case: DECISION-SUPPORT")
print("Causal claim: NO")
print("Future-performance guarantee: NO")

CLAIM REWRITE CHECK
Week-5 Precision@20: 65.0%
ML-09 5-fold Precision@20: 48.0%
ML-09 5-fold baseline Precision@20: 36.0%

Week-5 Precision@50: 72.0%
ML-09 5-fold Precision@50: 46.8%
ML-09 5-fold baseline Precision@50: 36.4%

Claim language: OBSERVED
Claim language: MEASURED
Claim language: DIRECTIONAL
Use case: DECISION-SUPPORT
Causal claim: NO
Future-performance guarantee: NO


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.